---
#  Vectorization 
---

In [1]:
%load_ext autoreload
%autoreload 2

import vec, tests



Many algorithms are using the same operation on many times on different data. This is usually done with a `for` loop, where each data point is operated on one after the other. But modern CPUs are capable of performing [SIMD](https://en.wikipedia.org/wiki/Single_instruction,_multiple_data), where multiple data are operated on at the same time. This process of performing the same operation simultaneously on different data is called _vectorization_. Even if not the complete data can be handeled by the CPU at the same time, it is often faster to use a vectorized operation.

While python is a language that is very easy to read and write, it is somewhat slow. Therefore, loops should be avoided as much as possible and be replaced with an call to a library that is implemented on a fast language. The exercise in this lecture will use _pytorch_.

---

## PyTorch

---

If we want to sum the elements of a tensor, the naive implementation would go over each element separately:

```
sum = 0
for element in tensor:
    sum += element
```

The vectorized version uses a function provided by the library:


```
sum = torch.sum(tensor)
```

Below, you can run a speed comparison of the different implementations, where the median along with the minimum and maximum measured time are marked. As your system runs other background tasks, the minium is the most important of these values. Note that the y-axis is in log scale!

In [2]:
tests.benchmark_sum()

---

## Vectorization Tips and Tricks

---

- Make sure your loop can be vectorized: To be vectorizable, the single datapoints must be accessible independently of the other loop iterations. An example of a _non_ vectorizable loop would be calculating the $n$-th Fibonacci number (defined as `f(n) = f(n-1) + f(n-2)`).
- Don't think of your data as sets $\{x_i\}$, but as vectors $x$ and matrices $X$. Generally avoid accessing single elements.
- If you need to access a subset, instead of using array indexing with the integer location, you can use bitmask. A bitmask is a tensor that has the same shape as a data tensor, but only holds booleans. They can be created by comparisons, e.g. `mask = data > threshold`. To then use these elements, simply do `data[mask]`. (You can still use integers to access a tensor, or even a tensor of longs for multiple elements.)
- Use [broadcasting](https://docs.pytorch.org/docs/stable/notes/broadcasting.html) to work with tensors of different shapes.
- Check the [torch docs](https://docs.pytorch.org/). Sometimes there are very specialized functions that one would not necessarily expect, e.g. [lerp](https://docs.pytorch.org/docs/stable/generated/torch.lerp.html) or [logcumsumexp](https://docs.pytorch.org/docs/stable/generated/torch.logcumsumexp.html).

Advanced:
- Use [gather](https://docs.pytorch.org/docs/stable/generated/torch.gather.html) and [scatter](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.scatter_.html) and variants like `scatter_reduce` to get and set elements for tensors with many dimensions.
- Check Einstein sum notation to use with [einsum](https://docs.pytorch.org/docs/stable/generated/torch.einsum.html). (But most operations are also implemented more explicitly by other pytorch functions. Check the docs!)

---
## Quicksort
---

The first task which we take a look at is _Quicksort_, which is a divide-and-conquer algorithm. It splits the given data with a pivot element into three subgroups, _less_, _equal_ and _greater_. These subgroups are than sorted recursively until only a single element is left.

```
def quick_sort_naive(arr: torch.Tensor ):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    less = []
    equal = []
    greater = []
    for element in arr:
        if element < pivot:
            less.append(element.view(1))
        elif element == pivot:
            equal.append(element.view(1))
        else:
            greater.append(element.view(1))
    less = torch.cat(less) if len(less) > 0 else torch.empty(0)
    equal = torch.cat(equal) if len(equal) > 0 else torch.empty(0)
    greater = torch.cat(greater) if len(greater) > 0 else torch.empty(0)
    return torch.cat([quick_sort_naive(less), equal, quick_sort_naive(greater)])
```

---

## **Task**: 

Implement `quick_sort_vec` in `vec.py` without any loops.

Take a look at this naive implementation `quick_sort_naive` for reference. Run the code here to check the implementation.


**Hints**:

- Consider `torch.cat`.
- Try to create a bitmask for each of the three subgroups which hold boolean comparison for each element of `arr`.


In [3]:
tests.test_sort(vec.quick_sort_vec)

Sorted


---

### Speed Test

If you have correctly implemented the vectorized function, check the speedup. The vecorized method should be ~ 5 times faster than the naive method. We also compare the vectorized method to a python native method which uses `list` as the main data structure, as well as the pytorchs `sort()` implementation.

In [4]:
tests.benchmark_sort().show()

You should see now that while vectorization leads to a big speedup, the python native code is actually faster. It is still not as fast as `torch.sort`, but why is that the case? The answer is that recursion and pytorch does not fit well. So avoid using many recursion in your pytorch code! Let's now look at scenarios that do not use recursions.

---
## Linear Model
---

The evolutionary predecessor of neural networks are linear models. For an input $x \in \mathbb{R}^D$, their parameters are a weight  $w \in \mathbb{R}^D$ and a bias  $b \in \mathbb{R}$. A prediction is than calculated as:

$$
\hat{y} = x \cdot w + b
$$

We will store all our datapoints as rows in a matrix $X \in \mathbb{R}^{N \times D}$, such that we expect our predictions to be a vector $\hat{y} \in \mathbb{R}^N$.

```
def linear_prediction(X: torch.Tensor, w: torch.Tensor, b: torch.Tensor):
    y = torch.empty(X.shape[0])
    for i in range(X.shape[0]):
        y[i] = torch.dot(w,X[i]) + b
    return y
```
---

## **Task**: 

Implement `linear_predicition_vec` in `vec.py`. Take a look at `linear_prediction` for reference. Run the code here to check the implementation.

**Hints:**

- Try to find a mathematical operation which computes many dot products at the same time.
- To add `b` to all elements at the same time, try broadcasting.

In [5]:
tests.test_linear(vec.linear_prediction_vec)

The prediction is correct


---

### Speed Test

Once again, we will check the speed of our implementation. This is a task where pytorch excels, such that the speedup should be about ~10 times. You should also easily beat the python native implementation.

In [6]:
tests.benchmark_linear().show()

---
## Distance calculations
---

For a data matrix $X \in \mathbb{R}^{N \times D}$, we want to calculate the distance matrix $D \in \mathbb{R} ^{N \times N}$ which holds the pairwise distances between the data points. The euclidean distance is defined as:

$$
d(x, y) = \sqrt{  \sum_{i=1}^D (x_i - y_i)^2 }
$$

```
def calculate_distances(X: torch.Tensor):
    N, D = X.shape
    dist = torch.zeros(N, N)
    for i in range(N):
        for j in range(i, N):
            diff = X[i, :] - X[j, :]
            dist[i, j] = diff.pow_(2).sum(dim=0)
            dist[i, j].sqrt_()

    dist = dist + dist.T
    return dist
```

---

## **Task**: 

Implement `distance_calculation_vec` in `vec.py` without any loops. Take a look at `distance_calculation` for reference. Run the code here to check the implementation.


**Hints**:
- The loop version uses a trick that only calculates a upper triangular matrix and them uses the symmetry $d(x, y) = d(y, x)$. Vectorization can't really handle this, so calculate all distances simultaneously.
- First try to implement this for $D=1$, before increasing it to more dimensions.
- Think about which shape `diff` should have in a vectorized version.
- You can use `tensor.squeeze` and `tensor.unsqueeze` to modify the dimensions of a tensor.

In [7]:
tests.test_distance(vec.calculate_distance_vec)

The distances are correct


---

### Speed Test

Once again, we will check the speed of our implementation. The speedup should be about ~100 times. But there is still a more efficient method, which uses an identity for the Euclidean distance $d(x, y) = |x| + |y| - 2 \langle x , y \rangle $. This method is even on par with PyTorchs implemented distance function `torch.cdist()`. Check `calculate_distance_identity()` if you are interested.

In [8]:
tests.benchmark_distance().show()